# 🏛️ vLLM 架构总览 — 设计哲学与系统全景

**前置阅读**：建议先读完 `theory/` 中的 "GPU 显存布局" 和 "混合推理优化"，理解 KV Cache 管理和各框架的对比。

**本文目标**：建立 vLLM 系统的完整心智模型——从设计哲学到代码结构。读完这篇你会理解：

- vLLM 解决的核心问题（不是"更快"，而是"更高效地使用显存"）
- 四大核心子系统：Block Manager、Scheduler、Worker、API Server
- 从 API 请求到 token 生成的完整数据流
- 源码目录结构和阅读入口

## 1. vLLM 解决的核心问题

### 1.1 不是"更快"，而是"更高效地用显存"

vLLM 的核心洞察写在论文标题里：**Efficient Memory Management for Large Language Model Serving**。

```
问题: 2023 年之前的推理框架 (FasterTransformer, TGI 早期版本等)
  KV Cache 采用静态预分配 → 严重的显存浪费

  示例: 1 个请求, max_tokens=2048
  静态分配: 预分配 2048 个位置的 KV Cache → ~1 GB
  实际使用: 请求在第 312 个 token 就结束了 → 只用了 ~150 MB
  浪费: 850 MB (85%!)

  → 同时只能服务少量请求 (因为每个请求的 KV Cache 预留过大)
  → GPU 利用率低

vLLM 的解法: PagedAttention
  类比操作系统的虚拟内存:
    物理内存 → GPU 显存
    虚拟地址 → 逻辑 KV Cache 位置
    页表     → Block Table
    页面     → KV Cache Block

  效果:
    同一个请求: 按需分配 blocks, 只占 ~150 MB
    → 同样的显存可以服务 5-6x 的并发请求
    → 论文号称 ~4x 吞吐提升 (实际场景差异很大)
```

### 1.2 历史定位

```
2023 年之前的推理方案:
  PyTorch model.generate()
    → 简单但极其低效
  FasterTransformer (NVIDIA)
    → 针对固定模型优化, 但不灵活, 不支持动态 batching
  TGI (HuggingFace)
    → 最早的 dynamic batching, 但 KV Cache 管理粗糙

2023 年 vLLM 的出现:
  → 把 "操作系统虚拟内存" 的思想搬到 KV Cache 管理
  → 首次实现了真正高效的动态显存分配
  → 成为开源推理框架的事实标准

2024 年竞争格局:
  vLLM vs SGLang vs TensorRT-LLM
  → vLLM: 生态最完善, 默认选择
  → SGLang: 前缀共享极致, Agent 场景优势
  → TensorRT-LLM: 编译优化, 单模型极致性能
```

## 2. 四大核心子系统

```
┌─────────────────────────────────────────────────────────────────┐
│                        vLLM 系统架构                              │
├─────────────────────────────────────────────────────────────────┤
│                                                                   │
│  ┌───────────────────┐                                            │
│  │   API Server       │  ← FastAPI / OpenAI-compatible             │
│  │   (vllm/entrypoints)│    接收请求, 流式响应, tokenization        │
│  └────────┬──────────┘                                            │
│           │                                                       │
│  ┌────────┴──────────┐                                            │
│  │   Scheduler        │  ← 调度核心: 决定谁先跑、谁被抢占           │
│  │   (vllm/core/)     │    Continuous batching 的逻辑在这里         │
│  └────────┬──────────┘                                            │
│           │                                                       │
│  ┌────────┴──────────┐                                            │
│  │   Block Manager    │  ← 显存管理核心: PagedAttention 的实现      │
│  │   (vllm/core/)     │    Block 的分配/回收/复制/共享             │
│  └────────┬──────────┘                                            │
│           │                                                       │
│  ┌────────┴──────────┐                                            │
│  │   Worker(s)        │  ← GPU 执行: 真正跑模型前向传播            │
│  │   (vllm/worker/)   │    ModelRunner, GPUModelRunner             │
│  └────────┬──────────┘                                            │
│           │                                                       │
│  ┌────────┴──────────┐                                            │
│  │   Model Executor   │  ← 模型后端: 调用 PyTorch/TRT/CUDA graph  │
│  │   (vllm/model_executor/) │                                     │
│  └───────────────────┘                                            │
│                                                                   │
└─────────────────────────────────────────────────────────────────┘
```

### 2.1 API Server
- FastAPI 框架, OpenAI-compatible endpoints (`/v1/chat/completions`, `/v1/completions`)
- 负责请求解析、tokenization、SSE 流式响应
- 不涉及 GPU 操作, 可以水平扩展

### 2.2 Scheduler
- **整个系统的"大脑"**
- 维护三个队列: WAITING / RUNNING / SWAPPED
- 每次 forward 前做调度决策: 哪些请求进 batch? 哪些被抢占?
- 实现了 continuous batching 的核心逻辑

### 2.3 Block Manager
- **PagedAttention 的实现**
- 管理 GPU 显存中的 KV Cache blocks
- 支持 block 的分配 (allocate)、释放 (free)、复制 (copy-on-write for prefix sharing)、交换 (swap to CPU)
- 类比 Linux 的 buddy allocator 或 slab allocator

### 2.4 Worker
- 持有模型权重, 执行 forward pass
- 支持 tensor parallel (多卡) 和 pipeline parallel
- 负责构建 CUDA graph (减少 kernel launch overhead)
- ModelRunner 是实际的执行单元

## 3. 完整请求生命周期

```
时间线: 一个 chat completion 请求的完整旅程

Step 1: 请求到达
  POST /v1/chat/completions
  {"model": "llama-3", "messages": [{"role": "user", "content": "你好"}]}
      │
      ▼
  API Server: 解析请求 → Tokenizer → input_ids
      │
      ▼
  Scheduler: 请求加入 WAITING queue

Step 2: 调度 (Scheduler)
  检查条件:
  - 显存是否足够? (Block Manager 询问)
  - 当前 batch 是否小于 max_num_seqs?
  - 请求是否等了太久需要优先处理?
      │
      ▼
  决定: 将请求从 WAITING → RUNNING

Step 3: Prefill (Worker)
  Block Manager 为请求分配 KV Cache blocks
  模型前向传播: [prompt tokens] → KV Cache + first token
      │
      ▼
  返回第一个 token → API Server 流式推给用户

Step 4: Decode Loop (Scheduler + Worker 交替)
  ┌─────────────────────────────────────────────┐
  │ 每步循环:                                    │
  │   Scheduler:                                │
  │     - 检查 running 请求谁完成了?              │
  │     - 有新请求可以加入 batch 吗?              │
  │     - 显存紧张需要抢占吗?                     │
  │     - 构建当前 step 的 batch                 │
  │   Worker:                                   │
  │     - 执行 batch forward (所有 running 请求)  │
  │     - 等待 GPU 返回 next tokens              │
  │     - 推理结果 → Scheduler (更新状态)        │
  │     - 新 token → API Server (推送给用户)     │
  └─────────────────────────────────────────────┘

Step 5: 完成
  请求生成了 stop token 或达到 max_tokens
  → Block Manager 释放该请求的所有 blocks
  → Scheduler 从 RUNNING 移除
  → API Server 发送 [DONE]
```

### 关键代码入口 (源码阅读指南)

```python
# 1. 启动入口
vllm/entrypoints/openai/api_server.py  # API Server 启动
vllm/engine/llm_engine.py              # LLMEngine: 核心编排器

# 2. 调度核心
vllm/core/scheduler.py                 # Scheduler: 调度逻辑
# 关键方法: _schedule() → 返回 SchedulerOutput

# 3. 显存管理
vllm/core/block_manager.py             # BlockSpaceManager
vllm/core/block/                        # Block 数据结构的实现
# 关键类: BlockTable, NaiveBlock, PrefixCachingBlock

# 4. 模型执行
vllm/worker/worker.py                  # Worker: 持有模型, 执行 forward
vllm/worker/model_runner.py            # ModelRunner: 实际的 forward pass
# 关键方法: execute_model() → 返回采样后的 token IDs

# 5. Attention 后端
vllm/attention/                         # 各种 attention 实现
vllm/attention/ops/paged_attn.py       # PagedAttention kernel
```

## 4. vLLM 的独特设计决策

### 4.1 "Everything is a SequenceGroup"

vLLM 把所有请求（包括多模态、工具调用的多轮对话）统一抽象为 `SequenceGroup`：

```python
# 概念模型 (简化)
class SequenceGroup:
    request_id: str
    seqs: List[Sequence]          # 一个请求可能对应多个 sequence
    sampling_params: SamplingParams
    arrival_time: float
    lora_request: Optional[LoRARequest]  # LoRA adapter

class Sequence:
    seq_id: int
    token_ids: List[int]          # 已生成的 tokens
    status: SequenceStatus        # WAITING / RUNNING / FINISHED
    block_table: BlockTable       # KV Cache 的映射表
    
# 多模态: 增加了 MultiModalData
# 工具调用: 通过 add_request() 注入 tool result 作为新的 prompt
```

### 4.2 与 HuggingFace 的深度集成

```python
# vLLM 默认使用 HuggingFace 的模型权重格式
from vllm import LLM

# 直接传 HuggingFace model ID
llm = LLM(model="meta-llama/Llama-3-8B-Instruct")
# vLLM 自动下载、加载、转换为自己的内部格式

# 支持任何 HuggingFace 兼容的模型:
#   LLaMA, Mistral, Qwen, Gemma, Phi, Falcon...
# 只要实现了 HuggingFace 的标准接口
```

### 4.3 Tensor Parallel 的开箱即用

```bash
# 单机多卡推理
vllm serve meta-llama/Llama-3-70B-Instruct     --tensor-parallel-size 4  # 自动切分到 4 张 GPU

# vLLM 使用 Ray 或 multiprocessing 管理多 worker
# 通过 NCCL 做跨 GPU 的 all-reduce (attention) 和 all-gather (MLP)
```